# 01 — Data Prep & Quality Checks

**Business question this repo answers:** where is this subscription business losing revenue,
which levers actually retain subscribers, and what should the company do about it? This is a
retention and pricing analysis, not a churn-prediction leaderboard exercise — the predictive
model in notebook 05 is one section near the end, not the point.

**Scope:** only `transactions_v2.csv`, `members_v3.csv`, and `train_v2.csv`. `user_logs_v2.csv`
(30GB+ of raw listening events) is deliberately excluded — it's not needed for a
billing/retention analysis and would make this repo impossible to run on a laptop.

**Data boundaries** (see main README for the full statement):
- Coverage is roughly January 2015 to March 2017.
- Training labels (`train_v2.csv`) cover subscriptions expiring February 2017.
- Churn is defined as: no renewal within 30 days of membership expiry.
- All prices are New Taiwan dollars (NT$).

This notebook: loads the three files, checks them for the data-quality issues this dataset is
known to have, reconstructs per-user membership periods (the base unit every later notebook
builds on), and caches the result.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import data_load, cohorts, plotting

plotting.set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
transactions = data_load.load_transactions()
members = data_load.load_members()
train = data_load.load_train()

print(f"transactions_v2: {len(transactions):,} rows, {transactions['msno'].nunique():,} unique users")
print(f"members_v3:      {len(members):,} rows")
print(f"train_v2:        {len(train):,} rows, churn rate {train['is_churn'].mean():.4f}")

### Coverage check
Verify the stated Jan 2015 – Mar 2017 boundary against the actual data.

In [ ]:
print("transaction_date range:        ", transactions["transaction_date"].min().date(), "->", transactions["transaction_date"].max().date())
print("membership_expire_date range:  ", transactions["membership_expire_date"].min().date(), "->", transactions["membership_expire_date"].max().date())
print("registration_init_time range:  ", members["registration_init_time"].min().date(), "->", members["registration_init_time"].max().date())

### Data quality: members_v3

`bd` (age) is self-reported and known to contain implausible values in this dataset — zeros,
negative numbers, and triple-digit ages. We flag these rather than silently dropping or
imputing them, since segment-level analysis later needs to visibly carve out an "unknown age"
bucket rather than quietly biasing the sample.

In [ ]:
implausible_age = ~members["bd"].between(10, 90)
print(f"{implausible_age.mean():.2%} of members have bd outside a plausible 10-90 range")
print(members.loc[implausible_age, "bd"].value_counts().head(10))

print("\ngender missing rate:", round(members["gender"].isna().mean(), 4))
print(members["registered_via"].value_counts(normalize=True).round(3))

We keep the raw `bd` column as-is (no imputation) and add an explicit `age_valid` flag downstream instead of guessing at true ages.

### Data quality: transactions_v2

In [ ]:
dupe_rows = transactions.duplicated().sum()
print(f"exact duplicate rows: {dupe_rows:,}")

zero_price = (transactions["plan_list_price"] == 0).mean()
print(f"plan_list_price == 0: {zero_price:.4%} of rows")

overpaid = (transactions["actual_amount_paid"] > transactions["plan_list_price"]).mean()
print(f"actual_amount_paid > plan_list_price: {overpaid:.4%} of rows")

print(transactions["payment_plan_days"].value_counts(normalize=True).round(3).head(10))
print(transactions["payment_method_id"].nunique(), "distinct payment methods")

`plan_list_price == 0` rows are real (promotional/trial transactions) and are kept, but excluded from discount-depth calculations in `src/pricing.py` — a discount percentage off a $0 list price is undefined, not zero.

### Building membership periods

See the `src/cohorts.py` module docstring for the full method. Short version: sort each user's
transactions chronologically and label each period `renewed` (another transaction started
within 30 days of this one's expiry), `voluntary_cancel` (`is_cancel` flagged on this
transaction), `lapsed_no_renewal` (expired, not flagged, never renewed), or `censored` (too
close to the data cutoff to know yet — excluded from renewal/lapse rates everywhere downstream).

In [ ]:
periods = cohorts.build_membership_periods(transactions)
members_cohort = cohorts.assign_cohort(members)

print(periods["outcome"].value_counts(normalize=True).round(3))
print(f"\n{periods['censored'].sum():,} periods ({periods['censored'].mean():.2%}) are censored.")

### Cache intermediate tables
`data/` is gitignored — this is a local speed cache. Delete it and rerun this notebook to rebuild.

In [ ]:
interim_dir = Path("../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)

periods.to_parquet(interim_dir / "periods.parquet")
members_cohort.to_parquet(interim_dir / "members_cohort.parquet")
train.to_parquet(interim_dir / "train.parquet")

print("Cached:", [p.name for p in interim_dir.glob("*.parquet")])

### Summary

- Loaded the three approved files only; row counts and coverage printed above.
- `bd` has known data-quality issues — flagged via `age_valid`, never silently cleaned.
- Membership periods reconstructed with an explicit right-censoring rule so recent activity
  doesn't get miscounted as churn just because there hasn't been time to observe a renewal yet.
- Every later notebook loads `periods.parquet` / `members_cohort.parquet` from this cache
  rather than rebuilding from raw transactions.